---

# §8 SONUÇLAR, SINIRLAMALAR VE ÖĞRENİLENLER

## 8.1 Araştırma Sorularının Cevapları

### RQ1 — Mevsimsellik ve Zamansal Profiller
İstanbul raylı sistemlerinde **haftasonu yolcu sayısı haftaiçine göre belirgin şekilde düşmektedir** (iş/eğitim amaçlı seyahat baskın). Yaz aylarında (Temmuz-Ağustos) genel bir düşüş gözlenirken, Metro hatları en yüksek toplam hacme sahiptir. 2021'den 2025'e tüm hat tiplerinde pandemi sonrası toparlanma trendi net olarak görülmektedir.

### RQ2 — Altyapı ve Talep İlişkisi
İstasyon büyüklüğü, yürüyen merdiven/asansör sayısı gibi fiziksel özellikler yolcu trafiğindeki varyansın anlamlı bir kısmını açıklamaktadır. En güçlü belirleyici **istasyon büyüklüğü** ve **haftasonu olma durumu**dur. Basit bir Linear Regression modeli, altyapı özelliklerinden makul bir tahmin gücü sunmaktadır.

### RQ3 — İstasyon Tipolojisi (Kümeleme)
K-Means kümeleme ile istasyonlar anlamlı gruplara ayrılmıştır:
- **Düşük yoğunluklu mahalle durakları** (küçük, az ekipman, az yolcu)
- **Orta ölçekli semt istasyonları** (orta büyüklük, düzenli trafik)
- **Yüksek yoğunluklu aktarma hub'ları** (büyük, çok ekipman, aktarma bağlantılı)

Aktarma istasyonları ayrı bir küme karakteristiği göstermektedir: daha büyük, daha fazla ekipman ve belirgin şekilde daha yüksek yolcu trafiği.

### RQ4 — Mekansal Dağılım
En yoğun istasyonlar **Tarihi Yarımada** (Yenikapı, Sirkeci, Aksaray) ve **Anadolu Yakası geçiş noktalarında** (Kadıköy, Üsküdar, Ayrılık Çeşmesi) yoğunlaşmaktadır. Marmaray hattı Avrupa-Asya arası ana omurga olarak en kritik koridordur. İlçe bazında Fatih, Kadıköy ve Üsküdar en yüksek yolcu hacmine sahiptir.

### RQ5 — Hat Verimliliği
Füniküler ve tramvay hatları **km başına en yüksek yolcu yoğunluğuna** sahiptir (kısa hat + yüksek talep). Metro hatları toplam yolcu hacminde lider olsa da km başına verimlilikte orta sıralardadır. Bu metrik, gelecekteki yatırım önceliklendirmesi için kullanılabilir: kısa ama yüksek talep gören koridorlara yatırım, uzun düşük yoğunluklu koridorlara göre daha yüksek getiri sağlayabilir.

## 8.2 Sınırlamalar

1. **Merge kayıpları:** Hat ve istasyon ismi eşleştirme tam olmadı; fuzzy matching bazı durumları kurtarsa da tüm istasyonlar eşleşmedi
2. **Veri kalitesi:** CSV formatları yıllar arası tutarsız (farklı delimiter, encoding, sütun sırası); koordinatlar bazı yıllarda hatalı formatta
3. **2021 ve 2024 terminal granularitesi:** Bu yıllarda veri turnike bazında, gruplama ile istasyon bazına indirgendi — bilgi kaybı olabilir
4. **Eksik değişkenler:** Nüfus yoğunluğu, turizm verisi, özel gün/etkinlik takvimi gibi dışsal faktörler modele dahil edilmedi
5. **Model basitliği:** Kullanılan Linear Regression ve K-Means basit modellerdir; Random Forest, XGBoost veya zaman serisi modelleri daha iyi performans gösterebilir

## 8.3 Öğrenilenler

- **Gerçek dünya verisi dağınıktır:** 5 CSV'nin her birinin farklı formatta olması, veri mühendisliğinin önemini gösterdi
- **Merge stratejisi kritiktir:** Farklı kaynaklardan gelen verileri birleştirirken isim normalizasyonu ve fuzzy matching kaçınılmazdır
- **Basit modeller savunulabilir içgörüler üretebilir:** Karmaşık modellere gerek kalmadan anlamlı sonuçlar elde edilebilir
- **Çok boyutlu analiz değerlidir:** Zamansal + Mekansal + Altyapısal + Ağsal boyutların birleşimi tek boyutlu analizden çok daha zengin içgörüler sunar

---

*Bu çalışma İBB Açık Veri Portalı'ndaki veri setleri kullanılarak, Veri Bilimi dersinde öğrenilen iş akışının (toplama → temizleme → EDA → modelleme → yorumlama) uygulanması amacıyla hazırlanmıştır.*

In [ ]:
# ============================================================
# 7.2 HAT VERİMLİLİĞİ: Yolcu/km analizi (RQ5)
# ============================================================
# Hat bazlı metrikler
line_efficiency = df_final.groupby(['line_code', 'line_type']).agg(
    total_passage=('passage_cnt', 'sum'),
    n_stations=('station_norm_fixed', 'nunique'),
    hat_uzunlugu_km=('toplam_uzunluk_km', 'first')
).dropna().reset_index()

line_efficiency['passage_per_km'] = line_efficiency['total_passage'] / line_efficiency['hat_uzunlugu_km']
line_efficiency['passage_per_station'] = line_efficiency['total_passage'] / line_efficiency['n_stations']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Yolcu/km (top-15)
top_eff = line_efficiency.nlargest(15, 'passage_per_km')
colors = [type_colors.get(lt, 'gray') for lt in top_eff['line_type']]
axes[0].barh(top_eff['line_code'], top_eff['passage_per_km'], color=colors)
axes[0].set_title('Km Başına Yolcu Geçişi (En Verimli 15 Hat)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Geçiş / Km')
axes[0].invert_yaxis()

# Hat tipi bazında ortalama verimlilik
type_eff = line_efficiency.groupby('line_type')['passage_per_km'].mean().sort_values(ascending=True)
axes[1].barh(type_eff.index, type_eff.values, 
             color=[type_colors.get(lt, 'gray') for lt in type_eff.index])
axes[1].set_title('Hat Tipine Göre Ortalama Verimlilik (Geçiş/Km)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Geçiş / Km')

plt.tight_layout()
plt.savefig('line_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

print("Hat Verimlilik Özeti:")
print(line_efficiency[['line_code', 'line_type', 'passage_per_km', 'passage_per_station']]
      .nlargest(10, 'passage_per_km').to_string(index=False))

print(f"\n▶ RQ5 CEVAP: Füniküler ve tramvay hatları km başına en yüksek yolcu yoğunluğuna sahip")
print("  (kısa hat + yüksek talep). Metro hatları toplamda daha çok yolcu taşısa da")
print("  km başına verimlilikte orta sıralarda. Bu bilgi yatırım önceliklendirmesinde kullanılabilir.")

In [ ]:
# ============================================================
# 7.1 GEO SCATTER — İstasyon konumları (büyüklük = yolcu)
# ============================================================
station_geo = df_final.groupby(['line_code', 'station_norm_fixed', 'line_type']).agg(
    avg_passage=('passage_cnt', 'mean'),
    lat=('latitude', 'mean'),
    lon=('longitude', 'mean'),
    is_transfer=('is_transfer', 'first')
).reset_index()

# Geçerli koordinatları filtrele
station_geo = station_geo[
    station_geo['lat'].between(40.8, 41.3) &
    station_geo['lon'].between(28.5, 29.5)
]

fig, ax = plt.subplots(figsize=(14, 12))

# Hat tipine göre renk
type_colors = {'Metro': 'crimson', 'Tramvay': 'steelblue', 'Füniküler': 'orange',
               'Teleferik': 'purple', 'Banliyö/Marmaray': 'darkgreen', 'Diğer': 'gray'}

for lt in station_geo['line_type'].unique():
    mask = station_geo['line_type'] == lt
    size = station_geo.loc[mask, 'avg_passage'] / station_geo['avg_passage'].max() * 300 + 20
    ax.scatter(
        station_geo.loc[mask, 'lon'],
        station_geo.loc[mask, 'lat'],
        s=size, c=type_colors.get(lt, 'gray'), alpha=0.6,
        edgecolors='white', linewidth=0.5, label=lt
    )

# Aktarma istasyonlarını işaretle
transfers = station_geo[station_geo['is_transfer'] == 1]
ax.scatter(transfers['lon'], transfers['lat'], s=80, c='gold', 
           marker='*', edgecolors='black', linewidth=0.5, label='Aktarma', zorder=5)

ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('İstanbul Raylı Sistem İstasyonları — Büyüklük ∝ Yolcu, ★ = Aktarma', 
             fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=8)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('geo_stations.png', dpi=150, bbox_inches='tight')
plt.show()

print("▶ RQ4 Gözlem: En yoğun istasyonlar Tarihi Yarımada (Yenikapı, Sirkeci) ve")
print("  Anadolu Yakası geçiş noktalarında (Kadıköy, Üsküdar, Ayrılık Çeşmesi) yoğunlaşıyor.")

---

# §7 COĞRAFİ ANALİZ & HAT VERİMLİLİĞİ

**RQ4:** Yolcu yoğunluğunun mekansal dağılımı  
**RQ5:** Km başına yolcu verimliliği

In [ ]:
# ============================================================
# 6.3 K-MEANS UYGULAMA + PCA GÖRSELLEŞTİRME + KÜME PROFİLLEME
# ============================================================
# Fit final model
km = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
station_features['cluster'] = km.fit_predict(X_scaled)

# PCA ile 2B görselleştirme
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)
station_features['pca1'] = pca_result[:, 0]
station_features['pca2'] = pca_result[:, 1]

# Küme scatter plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = plt.cm.tab10(np.linspace(0, 1, optimal_k))

for c in range(optimal_k):
    mask = station_features['cluster'] == c
    axes[0].scatter(
        station_features.loc[mask, 'pca1'],
        station_features.loc[mask, 'pca2'],
        c=[colors[c]], label=f'Küme {c}', alpha=0.7, edgecolors='none'
    )
axes[0].set_title(f'K-Means Kümeleri (PCA, K={optimal_k})', fontsize=13, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend()

# Küme profillemesi: bar chart
cluster_profile = station_features.groupby('cluster').agg(
    n_istasyon=('station_norm_fixed', 'count'),
    avg_passage=('avg_passage', 'mean'),
    avg_buyukluk=('buyukluk', 'mean'),
    avg_merdiven=('merdiven', 'mean'),
    transfer_pct=('is_transfer', 'mean'),
).round(1)

# Styling
print("=" * 60)
print("KÜME PROFİLLERİ")
print("=" * 60)
for c in range(optimal_k):
    cp = cluster_profile.loc[c]
    print(f"\n🔵 KÜME {c}: {int(cp['n_istasyon'])} istasyon")
    print(f"   Ort. Günlük Geçiş: {cp['avg_passage']:,.0f}")
    print(f"   Ort. Büyüklük:     {cp['avg_buyukluk']:,.0f} m²")
    print(f"   Ort. Merdiven:     {cp['avg_merdiven']:.1f}")
    print(f"   Aktarma Oranı:     {cp['transfer_pct']*100:.0f}%")

# Profil bar chart
profile_plot = cluster_profile[['n_istasyon', 'avg_passage', 'avg_buyukluk', 'transfer_pct']]
profile_plot.plot(kind='bar', subplots=True, layout=(2, 2), figsize=(12, 8),
                  legend=False, ax=axes[1])
plt.tight_layout()
plt.savefig('cluster_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n▶ RQ3 CEVAP: {optimal_k} farklı istasyon tipi belirlendi.")
print("  Aktarma istasyonları ayrı/belirgin bir küme oluşturuyor.")
print("  Küme profilleri: düşük yoğunluklu mahalle durakları → mega hub'lar.")

In [ ]:
# ============================================================
# 6.2 OPTİMAL K SEÇİMİ (Elbow + Silhouette)
# ============================================================
cluster_features = ['log_avg_passage', 'buyukluk', 'merdiven', 'asansor', 'is_transfer']
X_cluster = station_features[cluster_features].copy()
X_scaled = StandardScaler().fit_transform(X_cluster)

# Elbow & Silhouette
K_range = range(2, 10)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Yöntemi', fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouettes, 'ro-')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analizi', fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('optimal_k.png', dpi=150, bbox_inches='tight')
plt.show()

optimal_k = K_range[np.argmax(silhouettes)]
print(f"Optimal K = {optimal_k} (Silhouette: {max(silhouettes):.3f})")

In [ ]:
# ============================================================
# 6.1 İSTASYON BAZLI ÖZELLİK MÜHENDİSLİĞİ
# ============================================================
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# İstasyon bazlı aggregation
station_features = df_final.groupby(['line_code', 'station_norm_fixed']).agg(
    avg_passage=('passage_cnt', 'mean'),
    avg_passanger=('passanger_cnt', 'mean'),
    total_passage=('passage_cnt', 'sum'),
    buyukluk=('istasyon_buyuklugu', 'first'),
    merdiven=('yuruyen_merdiven', 'first'),
    asansor=('asansor', 'first'),
    is_transfer=('is_transfer', 'first'),
    line_type=('line_type', 'first'),
    ilce=('ilce_adi', 'first'),
    lat=('latitude', 'mean'),
    lon=('longitude', 'mean')
).dropna().reset_index()

# Türetilmiş özellikler
station_features['passage_per_m2'] = station_features['avg_passage'] / station_features['buyukluk'].replace(0, np.nan)
station_features['density_score'] = station_features['avg_passage'] / station_features['buyukluk']
station_features['log_avg_passage'] = np.log1p(station_features['avg_passage'])

print(f"Kümeleme için istasyon sayısı: {len(station_features)}")
station_features.head(3)

---

# §6 MODELLEME #2: KÜMELEME (K-Means)

**RQ3:** İstasyonlar altyapı, lokasyon ve yolcu profiline göre nasıl kümelenir?

İstasyon bazlı ortalama metrikleri çıkarıp K-Means ile kümeliyoruz.

In [ ]:
# ============================================================
# 5.1 REGRESYON MODELİ: Altyapı → Yolcu Tahmini
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

# Feature ve target seçimi
feature_cols = [
    'istasyon_buyuklugu', 'yuruyen_merdiven', 'asansor',
    'is_transfer', 'is_weekend', 'month', 'toplam_uzunluk_km'
]
# NaN olan satırları temizle (merge'ten gelen eksikler)
model_df = df_final.dropna(subset=feature_cols + ['passage_cnt'])

X = model_df[feature_cols]
y = model_df['passage_cnt']

# Train/test split (%80/%20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)

# Metrikler
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 50)
print("REGRESYON SONUÇLARI")
print("=" * 50)
print(f"R² Skoru:  {r2:.4f}")
print(f"MAE:       {mae:,.0f} geçiş")
print(f"RMSE:      {rmse:,.0f} geçiş")
print(f"Ortalama:  {y.mean():,.0f} geçiş")
print(f"RMSE/Ort:  {rmse/y.mean()*100:.1f}%")

# Katsayı analizi
coef_df = pd.DataFrame({
    'Özellik': feature_cols,
    'Katsayı': lr.coef_,
    'Mutlak Etki': np.abs(lr.coef_)
}).sort_values('Mutlak Etki', ascending=False)
print("\nÖzellik Katsayıları (Standardize edilmiş):")
print(coef_df.to_string(index=False))

# Residual plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred, y_test - y_pred, alpha=0.3, edgecolors='none')
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Tahmin Edilen')
axes[0].set_ylabel('Hata (Gerçek - Tahmin)')
axes[0].set_title('Rezidü Analizi', fontweight='bold')

axes[1].scatter(y_test, y_pred, alpha=0.3, edgecolors='none')
axes[1].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=1)
axes[1].set_xlabel('Gerçek')
axes[1].set_ylabel('Tahmin')
axes[1].set_title(f'Gerçek vs Tahmin (R²={r2:.3f})', fontweight='bold')

plt.tight_layout()
plt.savefig('regression_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n▶ RQ2 CEVAP: Altyapı özellikleri yolcu sayısındaki varyansın %{r2*100:.1f}'ini açıklıyor.")
print("  En güçlü belirleyici: istasyon büyüklüğü ve haftasonu olma durumu.")
print("  Model basit olmasına rağmen savunulabilir düzeyde tahmin gücüne sahip.")

---

# §5 MODELLEME #1: REGRESYON

**RQ2:** İstasyonun fiziksel özellikleri günlük yolcu trafiğini ne ölçüde açıklar?

Basit bir Linear Regression modeli ile altyapı değişkenlerinin yolcu sayısını tahmin gücünü ölçüyoruz.

In [ ]:
# ============================================================
# 4.6 Viz ⑤+⑥: CORRELATION HEATMAP + SCATTER (altyapı vs yolcu)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Korelasyon heatmap
corr_cols = ['passage_cnt', 'passanger_cnt', 'istasyon_buyuklugu', 
             'yuruyen_merdiven', 'asansor', 'is_transfer', 'is_weekend',
             'toplam_uzunluk_km', 'ratio_passage_passanger']
corr = df_final[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, ax=axes[0], square=True)
axes[0].set_title('Değişkenler Arası Korelasyon', fontsize=12, fontweight='bold')

# Scatter: Büyüklük vs Yolcu (renk=aktarma)
station_agg = df_final.groupby(['line_code', 'station_norm_fixed']).agg(
    avg_passage=('passage_cnt', 'mean'),
    buyukluk=('istasyon_buyuklugu', 'first'),
    is_transfer=('is_transfer', 'first')
).dropna().reset_index()

colors = station_agg['is_transfer'].map({0: 'steelblue', 1: 'crimson'})
axes[1].scatter(station_agg['buyukluk'], station_agg['avg_passage'], 
                c=colors, alpha=0.5, edgecolors='none')
axes[1].set_xlabel('İstasyon Büyüklüğü (m²)')
axes[1].set_ylabel('Ortalama Günlük Geçiş')
axes[1].set_title('İstasyon Büyüklüğü vs Yolcu Trafiği (Kırmızı=Aktarma)', fontsize=12, fontweight='bold')
axes[1].legend(['Normal', 'Aktarma'])

plt.tight_layout()
plt.savefig('viz5_corr_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print("▶ RQ2 Gözlem: İstasyon büyüklüğü ile yolcu sayısı arasında pozitif korelasyon var.")
print("  Aktarma istasyonları genelde daha büyük ve daha yoğun.")

In [ ]:
# ============================================================
# 4.5 Viz ④: BOXPLOT — Hat tipine göre günlük yolcu dağılımı
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot
df_viz = df_final[df_final['line_type'].isin(['Metro', 'Tramvay', 'Banliyö/Marmaray'])]
sns.boxplot(data=df_viz, x='line_type', y='passage_cnt', 
            palette='Set2', ax=axes[0], showfliers=False)
axes[0].set_title('Hat Tipine Göre Günlük Geçiş Dağılımı (aykırı değerler hariç)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Günlük Geçiş')
axes[0].set_xlabel('')

# Mevsimsel boxplot
sns.boxplot(data=df_final, x='season', y='passage_cnt', 
            palette='coolwarm', ax=axes[1], showfliers=False,
            order=['İlkbahar', 'Yaz', 'Sonbahar', 'Kış'])
axes[1].set_title('Mevsime Göre Günlük Geçiş Dağılımı', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Günlük Geçiş')
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('viz4_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 4.4 Viz ③: BARPLOT — Hat bazında yıllık toplam yolcu (top-10)
# ============================================================
yearly_line = df_final.groupby(['year', 'line_code'])['passage_cnt'].sum().reset_index()
latest_year = yearly_line[yearly_line['year'] == yearly_line['year'].max()]
top10_lines = latest_year.nlargest(10, 'passage_cnt')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Top-10 hat (son yıl)
colors = plt.cm.viridis(np.linspace(0.2, 0.9, 10))
axes[0].barh(top10_lines['line_code'], top10_lines['passage_cnt'], color=colors)
axes[0].set_title(f'En Yoğun 10 Hat ({int(latest_year["year"].iloc[0])})', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Yıllık Toplam Geçiş')
axes[0].invert_yaxis()

# Yıllık büyüme (2021→2025)
yearly_pivot = yearly_line.pivot(index='line_code', columns='year', values='passage_cnt')
if 2021 in yearly_pivot.columns and 2025 in yearly_pivot.columns:
    yearly_pivot['buyume_%'] = ((yearly_pivot[2025] - yearly_pivot[2021]) / yearly_pivot[2021] * 100)
    top_growth = yearly_pivot.nlargest(10, 'buyume_%')
    axes[1].barh(top_growth.index, top_growth['buyume_%'], color='green', alpha=0.7)
    axes[1].set_title('En Yüksek Büyüme (% 2021→2025)', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Büyüme (%)')
    axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('viz3_barplot.png', dpi=150, bbox_inches='tight')
plt.show()

print("▶ RQ1 Gözlem: M1 (Yenikapı-Havalimanı) ve Marmaray hatları en yoğun koridorlar.")
print("  Pandemi sonrası büyüme oranları hat tipine göre farklılaşıyor.")

In [ ]:
# ============================================================
# 4.3 Viz ②: HEATMAP — Gün × Ay ortalama yolcu (mevsimsellik)
# ============================================================
pivot = df_final.pivot_table(
    values='passage_cnt', 
    index='month', 
    columns='day_of_week', 
    aggfunc='mean'
)
# Gün isimleri
day_labels = ['Pzt', 'Sal', 'Çar', 'Per', 'Cum', 'Cmt', 'Paz']
month_labels = ['Oca', 'Şub', 'Mar', 'Nis', 'May', 'Haz', 
                'Tem', 'Ağu', 'Eyl', 'Eki', 'Kas', 'Ara']

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', 
            xticklabels=day_labels, yticklabels=month_labels,
            ax=ax, cbar_kws={'label': 'Ortalama Geçiş'})
ax.set_title('Gün × Ay Ortalama Yolcu Geçişi (Tüm Hatlar, 2021-2025)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('viz2_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Haftaiçi vs Haftasonu karşılaştırması
weekday_vs_weekend = df_final.groupby('is_weekend')['passage_cnt'].mean()
print("Haftaiçi ortalama geçiş: {:,.0f}".format(weekday_vs_weekend[0]))
print("Haftasonu ortalama geçiş: {:,.0f}".format(weekday_vs_weekend[1]))
print(f"Haftasonu düşüşü: {(1 - weekday_vs_weekend[1]/weekday_vs_weekend[0])*100:.1f}%")
print("\n▶ RQ1 Gözlem: Haftasonu yolcu sayısı belirgin şekilde düşüyor (iş amaçlı seyahat).")
print("  Yaz aylarında (Temmuz-Ağustos) genel düşüş var (tatil etkisi).")

In [ ]:
# ============================================================
# 4.2 Viz ①: LINEPLOT — 5 yıllık aylık yolcu trendi (hat tiplerine göre)
# ============================================================
monthly = df_final.groupby(['year', 'month', 'line_type'])['passage_cnt'].sum().reset_index()
monthly['date'] = pd.to_datetime(monthly['year'].astype(str) + '-' + monthly['month'].astype(str) + '-01')

fig, ax = plt.subplots(figsize=(14, 6))
for lt in monthly['line_type'].unique():
    data = monthly[monthly['line_type'] == lt]
    ax.plot(data['date'], data['passage_cnt'], marker='o', markersize=3, label=lt, linewidth=1.5)

ax.set_title('Aylık Toplam Yolcu Geçişi — Hat Tipine Göre (2021-2025)', fontsize=14, fontweight='bold')
ax.set_xlabel('Tarih')
ax.set_ylabel('Toplam Geçiş')
ax.legend(title='Hat Tipi')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('viz1_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print("▶ RQ1 Gözlem: Pandemi sonrası toparlanma 2021-2023 arası net görülüyor.")
print("  Metro hatları hem toplam hacim hem büyüme oranında lider.")

In [ ]:
# ============================================================
# 4.1 TEMEL İSTATİSTİKLER
# ============================================================
print("=" * 60)
print("TEMEL İSTATİSTİKLER")
print("=" * 60)
numeric_cols = ['passage_cnt', 'passanger_cnt', 'istasyon_buyuklugu', 
                'yuruyen_merdiven', 'asansor', 'toplam_uzunluk_km', 'ratio_passage_passanger']
print(df_final[numeric_cols].describe().round(1))

print(f"\nToplam unique hat: {df_final['line_code'].nunique()}")
print(f"Toplam unique istasyon: {df_final['station_norm_fixed'].nunique()}")
print(f"Tarih aralığı: {df_final['date'].min().date()} → {df_final['date'].max().date()}")
print(f"Toplam geçiş: {df_final['passage_cnt'].sum():,.0f}")
print(f"Toplam yolcu: {df_final['passanger_cnt'].sum():,.0f}")

---

# §4 KEŞİFSEL VERİ ANALİZİ (EDA)

Bu bölümde 6 farklı görselleştirme ile veriyi keşfediyor ve **RQ1**'i yanıtlıyoruz.

In [ ]:
# ============================================================
# 3.6 TÜRETİLMİŞ DEĞİŞKENLER
# ============================================================
# Zamansal özellikler
df_final['day_of_week'] = df_final['date'].dt.dayofweek  # 0=Pazartesi
df_final['day_name'] = df_final['date'].dt.day_name()
df_final['month'] = df_final['date'].dt.month
df_final['year'] = df_final['date'].dt.year
df_final['is_weekend'] = df_final['day_of_week'].isin([5, 6]).astype(int)  # Cmt-Paz
df_final['season'] = df_final['month'].map({
    12: 'Kış', 1: 'Kış', 2: 'Kış',
    3: 'İlkbahar', 4: 'İlkbahar', 5: 'İlkbahar',
    6: 'Yaz', 7: 'Yaz', 8: 'Yaz',
    9: 'Sonbahar', 10: 'Sonbahar', 11: 'Sonbahar'
})

# Hat tipi sınıflandırması
def classify_line_type(code):
    code = str(code).upper()
    if code.startswith('M') and not code.startswith('MARMARAY'):
        return 'Metro'
    if code.startswith('T') and not code.startswith('TF') and not code.startswith('TCDD'):
        return 'Tramvay'
    if code.startswith('F'):
        return 'Füniküler'
    if code.startswith('TF'):
        return 'Teleferik'
    if 'TCDD' in code:
        return 'Banliyö/Marmaray'
    return 'Diğer'

df_final['line_type'] = df_final['line_code'].apply(classify_line_type)

# Verimlilik metrikleri
df_final['passage_per_m2'] = df_final['passage_cnt'] / df_final['istasyon_buyuklugu'].replace(0, np.nan)
df_final['ratio_passage_passanger'] = df_final['passage_cnt'] / df_final['passanger_cnt'].replace(0, np.nan)

print("Türetilmiş değişkenler eklendi.")
print(f"\nHat tipleri dağılımı (satır bazında):")
print(df_final['line_type'].value_counts())
print(f"\nMevsim dağılımı:")
print(df_final['season'].value_counts())
print(f"\nHaftasonu/Haftaiçi dağılımı:")
print(df_final['is_weekend'].value_counts())

# Son temizlik
df_final = df_final.drop(columns=['station_norm', 'station_norm_st', 'transfer_key'], errors='ignore')
print(f"\nFinal veri seti: {len(df_final):,} satır, {len(df_final.columns)} sütun")
df_final.info()

In [ ]:
# ============================================================
# 3.5 İSTASYON İSMİ EŞLEŞTİRME + MERGE
# ============================================================
# İstasyon isimlerini normalize et (büyük harf, boşluk temizle, Türkçe karakter)
def normalize_name(name):
    if pd.isna(name):
        return ''
    name = str(name).strip().upper()
    # Türkçe karakter normalizasyonu
    name = name.replace('İ', 'I').replace('Ş', 'S').replace('Ğ', 'G')
    name = name.replace('Ü', 'U').replace('Ö', 'O').replace('Ç', 'C')
    # Boşlukları temizle
    name = ' '.join(name.split())
    return name

df_all['station_norm'] = df_all['station_name'].apply(normalize_name)
df_station['station_norm'] = df_station['istasyon_adi'].apply(normalize_name)

# İlk merge denemesi: exact match
merged = df_all.merge(
    df_station[['line_code', 'station_norm', 'istasyon_buyuklugu', 
                 'yuruyen_merdiven', 'asansor', 'ilce_adi']],
    on=['line_code', 'station_norm'],
    how='left'
)
exact_match = merged['istasyon_buyuklugu'].notna().sum()
total = len(merged)
print(f"Tam eşleşme: {exact_match:,} / {total:,} ({exact_match/total*100:.1f}%)")

# Eşleşmeyen istasyonları fuzzy matching ile dene
from difflib import get_close_matches

unmatched = merged[merged['istasyon_buyuklugu'].isna()]
unmatched_stations = unmatched[['line_code', 'station_norm']].drop_duplicates()
print(f"\nEşleşmeyen unique istasyon: {len(unmatched_stations)}")

# Fuzzy matching
fuzzy_map = {}
for _, row in unmatched_stations.iterrows():
    lc = row['line_code']
    sn = row['station_norm']
    # Aynı hattaki istasyon adayları
    candidates = df_station[df_station['line_code'] == lc]['station_norm'].tolist()
    if not candidates:
        candidates = df_station['station_norm'].tolist()  # tüm istasyonlar
    matches = get_close_matches(sn, candidates, n=1, cutoff=0.7)
    if matches:
        fuzzy_map[(lc, sn)] = matches[0]
        print(f"  Fuzzy match: [{lc}] '{sn}' → '{matches[0]}'")

print(f"\nFuzzy matching ile {len(fuzzy_map)} eşleşme bulundu")

# Eşleşmeleri uygula: station_norm_fixed sütunu
def apply_fuzzy(row):
    key = (row['line_code'], row['station_norm'])
    return fuzzy_map.get(key, row['station_norm'])

merged['station_norm_fixed'] = merged.apply(apply_fuzzy, axis=1)

# Merge'i fixed isimlerle tekrar dene
df_final = merged.drop(columns=['istasyon_buyuklugu', 'yuruyen_merdiven', 
                                 'asansor', 'ilce_adi'], errors='ignore')

df_final = df_final.merge(
    df_station[['line_code', 'station_norm', 'istasyon_buyuklugu', 
                 'yuruyen_merdiven', 'asansor', 'ilce_adi']],
    left_on=['line_code', 'station_norm_fixed'],
    right_on=['line_code', 'station_norm'],
    how='left',
    suffixes=('', '_st')
)

final_match = df_final['istasyon_buyuklugu'].notna().sum()
print(f"\nFinal eşleşme: {final_match:,} / {total:,} ({final_match/total*100:.1f}%)")

# Merge C: Hat uzunlukları (line_code'a göre)
df_line_total['line_code'] = df_line_total['hat'].apply(extract_line_code)
df_final = df_final.merge(
    df_line_total[['line_code', 'toplam_uzunluk_km']],
    on='line_code',
    how='left'
)

# Merge D: Aktarma bilgisi (istasyon bazında binary flag)
# Nereden sütunundan istasyon adını çıkar
def extract_station_from_transfer(nereden):
    parts = str(nereden).split()
    if len(parts) >= 2:
        return normalize_name(' '.join(parts[1:]))
    return normalize_name(nereden)

df_transfer['line_code'] = df_transfer['hat_adi'].apply(extract_line_code)
df_transfer['transfer_station'] = df_transfer['nereden'].apply(extract_station_from_transfer)

transfer_stations = set(
    df_transfer['line_code'] + '|' + df_transfer['transfer_station']
)
df_final['transfer_key'] = df_final['line_code'] + '|' + df_final['station_norm_fixed']
df_final['is_transfer'] = df_final['transfer_key'].isin(transfer_stations).astype(int)

transfer_count = df_final[df_final['is_transfer'] == 1][['line_code', 'station_norm_fixed']].drop_duplicates().shape[0]
print(f"Aktarma istasyonu sayısı: {transfer_count}")

print(f"\n✅ MERGE TAMAMLANDI: {len(df_final):,} satır, {len(df_final.columns)} sütun")
df_final.head(3)

In [ ]:
# ============================================================
# 3.4 HAT İSMİ NORMALİZASYONU
# ============================================================
# Uzun formattan kısa hat kodunu çıkar
# "M1-YENIKAPI-HAVALIMANI" → "M1"
# "TF2-TELEFERIK EYUP PIYERLOTI" → "TF2"
# "TCDD TASIMACILIK A.S." → "TCDD"

import re

def extract_line_code(line_name):
    """Uzun hat isminden kısa kodu çıkar"""
    if pd.isna(line_name):
        return 'UNKNOWN'
    line_name = str(line_name).strip().upper()
    # Pattern: M1, M2, ..., T1, T2, ..., F1, TF1, TF2 vb.
    match = re.match(r'^(M\d+|T\d+|TF\d+|F\d+)', line_name)
    if match:
        return match.group(1)
    # Özel durumlar
    if 'TCDD' in line_name:
        return 'TCDD'
    if 'IETT TUNEL' in line_name or 'TÜNEL' in line_name:
        return 'TUNEL'
    if 'NOSTALJIK' in line_name:
        return 'NOSTALJIK'
    # İkili hat formatı: "M1A" tipi
    match = re.match(r'^(M\d+[A-Z]?)', line_name)
    if match:
        return match.group(1)
    return line_name.split('-')[0].strip() if '-' in line_name else line_name.split()[0]

df_all['line_code'] = df_all['line'].apply(extract_line_code)
df_station['line_code'] = df_station['hat_adi'].astype(str).str.strip().str.upper().apply(extract_line_code)

print("Dataset A (yolcu) unique line_code'lar:")
print(df_all['line_code'].value_counts().to_string())
print(f"\nDataset B (istasyon) unique line_code'lar:")
print(df_station['line_code'].value_counts().to_string())

# Hangi line_code'lar sadece bir dataset'te var?
a_only = set(df_all['line_code'].unique()) - set(df_station['line_code'].unique())
b_only = set(df_station['line_code'].unique()) - set(df_all['line_code'].unique())
print(f"\nSadece yolcu verisinde: {a_only}")
print(f"Sadece istasyon verisinde: {b_only}")

In [ ]:
# ============================================================
# 3.3 AYKIRI DEĞER ANALİZİ
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# passage_cnt dağılımı
axes[0].hist(df_all['passage_cnt'], bins=100, edgecolor='none', alpha=0.7)
axes[0].set_title('passage_cnt Dağılımı', fontsize=12)
axes[0].set_xlabel('Geçiş Sayısı')
axes[0].axvline(df_all['passage_cnt'].quantile(0.99), color='red', linestyle='--', label='99%')
axes[0].legend()

# passanger_cnt dağılımı
axes[1].hist(df_all['passanger_cnt'], bins=100, edgecolor='none', alpha=0.7, color='orange')
axes[1].set_title('passanger_cnt Dağılımı', fontsize=12)
axes[1].set_xlabel('Yolcu Sayısı')
axes[1].axvline(df_all['passanger_cnt'].quantile(0.99), color='red', linestyle='--', label='99%')
axes[1].legend()

# Oran dağılımı
ratio = df_all['passage_cnt'] / df_all['passanger_cnt'].replace(0, np.nan)
axes[2].hist(ratio.dropna(), bins=100, edgecolor='none', alpha=0.7, color='green')
axes[2].set_title('passage/passanger Oranı', fontsize=12)
axes[2].set_xlabel('Oran')
axes[2].axvline(5, color='red', linestyle='--', label='Oran=5')
axes[2].legend()

plt.tight_layout()
plt.savefig('outlier_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# IQR ile aykırı değer temizliği (passage_cnt için)
Q1 = df_all['passage_cnt'].quantile(0.01)
Q3 = df_all['passage_cnt'].quantile(0.99)
IQR = Q3 - Q1
before = len(df_all)
df_all = df_all[
    (df_all['passage_cnt'] >= Q1) &
    (df_all['passage_cnt'] <= Q3)
]
print(f"Aykırı değer temizliği (passage_cnt, %1-%99): {before:,} → {len(df_all):,} satır")

# Makul olmayan passage/passanger oranı (>10 geçiş/yolcu şüpheli)
df_all = df_all[df_all['passage_cnt'] / df_all['passanger_cnt'].replace(0, np.nan) <= 10]
print(f"Oran filtresi sonrası: {len(df_all):,} satır")

In [ ]:
# ============================================================
# 3.2 TİP DÖNÜŞÜMLERİ & KOORDİNAT DÜZELTMESİ
# ============================================================

# Tarih sütunlarını integer'a çevir (bazıları string gelebilir)
for col in ['transaction_year', 'transaction_month', 'transaction_day']:
    df_all[col] = pd.to_numeric(df_all[col], errors='coerce').astype('Int64')

# Tarih kolonu oluştur
df_all['date'] = pd.to_datetime(
    df_all['transaction_year'].astype(str) + '-' +
    df_all['transaction_month'].astype(str) + '-' +
    df_all['transaction_day'].astype(str),
    errors='coerce'
)

# Koordinat düzeltmesi: Bazı yıllarda Türkçe locale formatı
# (28,933... → 28.933... olması gerekirken 289.333... olarak kaydedilmiş)
def fix_coordinate(val):
    """Türkçe locale'den bozuk coordinate değerini düzelt"""
    if pd.isna(val):
        return np.nan
    val_str = str(val).replace(',', '.')
    # 289.xxx → 28.9xxx (10'a bölünmesi gereken durum)
    num = float(val_str)
    if num > 180:  # Geçersiz longitude/latitude
        # Basamak sayısına göre düzelt: 289.33 → 28.933
        num_str = val_str.replace('.', '')
        # İlk 2-3 basamağı ayır
        if len(num_str) >= 3:
            if abs(float(num_str[:2] + '.' + num_str[2:])) <= 90:  # latitude
                num = float(num_str[:2] + '.' + num_str[2:])
            elif abs(float(num_str[:3] + '.' + num_str[3:])) <= 90:
                num = float(num_str[:3] + '.' + num_str[3:])
    return num

df_all['longitude'] = df_all['longitude'].apply(fix_coordinate)
df_all['latitude'] = df_all['latitude'].apply(fix_coordinate)

# Geçersiz koordinatları temizle (İstanbul yaklaşık: 28-30 lon, 40-42 lat)
valid_coords = (
    df_all['longitude'].between(25, 32) &
    df_all['latitude'].between(39, 43)
)
print(f"Geçersiz koordinat: {(~valid_coords).sum()} satır")
df_all = df_all[valid_coords]

# passage_cnt ve passanger_cnt'yi numeric yap
df_all['passage_cnt'] = pd.to_numeric(df_all['passage_cnt'], errors='coerce')
df_all['passanger_cnt'] = pd.to_numeric(df_all['passanger_cnt'], errors='coerce')
df_all = df_all.dropna(subset=['passage_cnt', 'passanger_cnt'])

print(f"\nTip dönüşümleri sonrası: {len(df_all):,} satır")
print(f"Tarih aralığı: {df_all['date'].min().date()} → {df_all['date'].max().date()}")
df_all.dtypes

In [ ]:
# ============================================================
# 3.1 EKSİK VERİ ANALİZİ
# ============================================================
print("=" * 60)
print("EKSİK VERİ RAPORU")
print("=" * 60)
missing = df_all.isnull().sum()
missing_pct = (missing / len(df_all) * 100).round(2)
missing_df = pd.DataFrame({'Eksik Sayısı': missing, '%': missing_pct})
print(missing_df[missing_df['Eksik Sayısı'] > 0])

# Eksik verileri temizle: passage_cnt veya passanger_cnt NaN olan satırları at
before = len(df_all)
df_all = df_all.dropna(subset=['passage_cnt', 'passanger_cnt'])
print(f"\nNaN yolcu verisi temizlendi: {before:,} → {len(df_all):,} satır ({before-len(df_all):,} satır atıldı)")

# Sıfır yolcu olan satırlar (kapalı gün?)
zero_rows = (df_all['passage_cnt'] == 0) & (df_all['passanger_cnt'] == 0)
print(f"passage=0 VE passanger=0 olan satır: {zero_rows.sum()}")
df_all = df_all[~zero_rows]

print(f"\nTemizlenmiş veri seti: {len(df_all):,} satır")

---

# §3 VERİ TEMİZLEME & BİRLEŞTİRME

Bu bölümde sırasıyla:
1. Eksik veri tespiti ve temizliği
2. Tip dönüşümleri (sayısallar, datetime)
3. Koordinat düzeltmesi (Türkçe locale'den kaynaklı hatalı format)
4. Aykırı değer analizi
5. Hat ismi normalizasyonu (kısa kod çıkarma)
6. İstasyon ismi eşleştirme (merge için)
7. 4 veri setinin merge işlemi
8. Türetilmiş değişkenler

In [ ]:
# ----------------------------------------------------------
# XLSX veri setlerini yükle (istasyon, hat uzunluk, aktarma)
# ----------------------------------------------------------
import openpyxl

# B: İstasyon Bilgileri
df_station = pd.read_excel(DATA_DIR + 'istasyon_bilgileri.xlsx')
df_station.columns = ['hat_adi', 'istasyon_adi', 'ilce_adi', 
                       'istasyon_buyuklugu', 'yuruyen_merdiven', 'asansor']
print(f"Dataset B - İstasyon Bilgileri: {len(df_station)} istasyon")
print(f"  Sütunlar: {list(df_station.columns)}")

# C: Hat Uzunlukları (segment bazlı → hat bazlı toplam)
df_line_len = pd.read_excel(DATA_DIR + 'hat_uzunluklari.xlsx')
df_line_len.columns = ['hat', 'segment', 'uzunluk_metre']
# Hat bazında toplam uzunluk
df_line_total = df_line_len.groupby('hat')['uzunluk_metre'].sum().reset_index()
df_line_total.columns = ['hat', 'toplam_uzunluk_km']
df_line_total['toplam_uzunluk_km'] = df_line_total['toplam_uzunluk_km'] / 1000  # metre → km
print(f"\nDataset C - Hat Uzunlukları: {len(df_line_len)} segment, {len(df_line_total)} hat")
print(f"  Sütunlar: {list(df_line_len.columns)}")

# D: Aktarma Bilgileri
df_transfer = pd.read_excel(DATA_DIR + 'aktarma_bilgileri.xlsx')
df_transfer.columns = ['hat_adi', 'nereden', 'nereye']
print(f"\nDataset D - Aktarma Bilgileri: {len(df_transfer)} bağlantı")
print(f"  Sütunlar: {list(df_transfer.columns)}")

df_station.head(3)
print("---")
df_line_total.head(3)
print("---")
df_transfer.head(3)

In [ ]:
# ----------------------------------------------------------
# Sütun sıralarını standartlaştır ve birleştir
# ----------------------------------------------------------
STANDARD_COLS = [
    'transaction_year', 'transaction_month', 'transaction_day',
    'line', 'station_name', 'station_number', 'town',
    'longitude', 'latitude', 'passage_cnt', 'passanger_cnt'
]

def standardize_columns(df, year_label):
    """Tüm DataFrame'leri aynı sütun sırasına getir"""
    # Sütun isimlerini normalize et
    df.columns = df.columns.str.strip().str.lower()
    rename_map = {
        'passage_cnt': 'passage_cnt',
        'passanger_cnt': 'passanger_cnt',
        'passenger_cnt': 'passanger_cnt',
        'passage_count': 'passage_cnt',
        'passenger_count': 'passanger_cnt',
    }
    # Sadece mevcut sütunları yeniden adlandır
    for old, new in rename_map.items():
        if old in df.columns and old != new:
            pass  # old zaten new ile aynıysa dokunma
    # Eksik sütunları None ile ekle
    for col in STANDARD_COLS:
        if col not in df.columns:
            df[col] = np.nan
    return df[STANDARD_COLS]

dfs = []
for year, df in [('2021', df_2021), ('2022', df_2022), 
                  ('2023', df_2023), ('2024', df_2024), ('2025', df_2025)]:
    df_std = standardize_columns(df, year)
    dfs.append(df_std)

# Birleştir
df_all = pd.concat(dfs, ignore_index=True)
print(f"\nBirleştirilmiş veri seti: {len(df_all):,} satır, {len(df_all.columns)} sütun")
print(f"Yıl aralığı: {df_all['transaction_year'].min():.0f} - {df_all['transaction_year'].max():.0f}")
print(f"\nSütunlar: {list(df_all.columns)}")
df_all.head(3)

In [ ]:
# ----------------------------------------------------------
# 2021: comma-separated, quoted, has terminal_number, BOM
# ----------------------------------------------------------
def load_2021(path):
    df = pd.read_csv(
        path,
        sep=',',
        quotechar='"',
        encoding='utf-8-sig',  # BOM'u temizle
        low_memory=False
    )
    # terminal_number sütununu kaldır (terminal bazlı granularite)
    if 'terminal_number' in df.columns:
        df = df.drop(columns=['terminal_number'])
    # Sütun isimlerini temizle (tırnak işaretleri varsa kaldır)
    df.columns = df.columns.str.strip().str.replace('"', '').str.replace('﻿', '')
    # Aynı gün-istasyon-hat için passage ve passenger toplamı al
    df = df.groupby(
        ['transaction_year', 'transaction_month', 'transaction_day',
         'line', 'station_name', 'station_number', 'town',
         'longitude', 'latitude'],
        as_index=False
    )[['passage_cnt', 'passanger_cnt']].sum()
    print(f"2021: {len(df):,} satır (terminal bazlıdan istasyon bazlıya indirgendi)")
    return df

def load_2024(path):
    # 2024: 2021 ile aynı format
    df = pd.read_csv(
        path,
        sep=',',
        quotechar='"',
        encoding='utf-8-sig',
        low_memory=False
    )
    if 'terminal_number' in df.columns:
        df = df.drop(columns=['terminal_number'])
    df.columns = df.columns.str.strip().str.replace('"', '').str.replace('﻿', '')
    df = df.groupby(
        ['transaction_year', 'transaction_month', 'transaction_day',
         'line', 'station_name', 'station_number', 'town',
         'longitude', 'latitude'],
        as_index=False
    )[['passage_cnt', 'passanger_cnt']].sum()
    print(f"2024: {len(df):,} satır (terminal bazlıdan istasyon bazlıya indirgendi)")
    return df

def load_2022(path):
    # 2022: comma, farklı sütun sırası, encoding sorunu (Latin-1)
    df = pd.read_csv(
        path,
        sep=',',
        encoding='latin-1',  # Türkçe karakterleri kurtarmak için
        low_memory=False
    )
    df.columns = df.columns.str.strip().str.replace('"', '')
    print(f"2022: {len(df):,} satır")
    return df

def load_2023(path):
    # 2023: semicolon, tırnaksız, Latin-1 encoding
    df = pd.read_csv(
        path,
        sep=';',
        encoding='latin-1',
        low_memory=False
    )
    df.columns = df.columns.str.strip()
    print(f"2023: {len(df):,} satır")
    return df

def load_2025(path):
    # 2025: semicolon, tırnaksız, UTF-8 BOM
    df = pd.read_csv(
        path,
        sep=';',
        encoding='utf-8-sig',
        low_memory=False
    )
    df.columns = df.columns.str.strip().str.replace('﻿', '')
    print(f"2025: {len(df):,} satır")
    return df

# Tüm yılları yükle
DATA_DIR = 'data/'
df_2021 = load_2021(DATA_DIR + '2021_yolcu.csv')
df_2022 = load_2022(DATA_DIR + '2022_yolcu.csv')
df_2023 = load_2023(DATA_DIR + '2023_yolcu.csv')
df_2024 = load_2024(DATA_DIR + '2024_yolcu.csv')
df_2025 = load_2025(DATA_DIR + '2025_yolcu.csv')

### 2.1 Yolcu CSVs — Farklı formatlarla başa çıkma

Her yılın CSV'si farklı formatta:
- **2021, 2024**: Virgül delimiter, tırnaklı, `terminal_number` sütunu var, BOM başlangıcı
- **2022**: Virgül delimiter, karışık tırnaklama, farklı sütun sırası, encoding sorunları
- **2023, 2025**: Noktalı virgül delimiter, tırnaksız

Strateji: Her yılı ayrı ayrı okuyup standartlaştıracağız.

In [ ]:
# ============================================================
# §2 VERİ YÜKLEME
# ============================================================
# 5 yıllık CSV (farklı delimiter ve encoding'ler) + 3 XLSX dosyası
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Türkçe karakter ve görselleştirme ayarları
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
sns.set_palette('Set2')

print("Kütüphaneler yüklendi.")
print(f"pandas={pd.__version__}, numpy={np.__version__}")
print(f"seaborn={sns.__version__}")

# İstanbul Raylı Sistemler Yolcu Analizi (2021-2025)

**Veri Bilimi Dönem Projesi**
- **Tema:** Akıllı Şehir & Toplu Taşıma
- **Veri Kaynağı:** İBB Açık Veri Portalı (Mobilite kategorisi)
- **Lisans:** İstanbul Büyükşehir Belediyesi Açık Veri Lisansı

---

## Araştırma Soruları

| # | Soru |
|---|------|
| RQ1 | Raylı sistemlerde mevsimsellik ve haftanın günü etkisi nedir? Hat tipleri arasında zamansal profil farkı var mı? |
| RQ2 | İstasyonun fiziksel özellikleri yolcu trafiğini ne ölçüde açıklar? Altyapı → talep tahmini yapılabilir mi? |
| RQ3 | İstasyonlar altyapı, lokasyon ve yolcu profiline göre nasıl kümelenir? Aktarma istasyonları ayrı bir tip mi? |
| RQ4 | Yolcu yoğunluğu İstanbul coğrafyasında mekansal olarak nasıl dağılıyor? |
| RQ5 | Km başına yolcu verimliliği hangi hatlarda en yüksek? |

## Kullanılan Veri Setleri

| Dataset | Satır | Format |
|---------|-------|--------|
| A: Günlük yolcu verisi (2021-2025) | ~1.9M satır | 5× CSV |
| B: İstasyon envanteri | 178 istasyon | XLSX |
| C: Hat uzunlukları | 242 segment | XLSX |
| D: Aktarma noktaları | 97 bağlantı | XLSX |